# Kernel-target alignment for thermal TFIM kernels

This notebook computes the standard centered kernel-target alignment for the normalized thermal TFIM IPR and HS kernel matrices.

For each inverse temperature $\beta$, kernel type, and system size, we load the normalized kernel

$$
\tilde K_{\mathrm{IPR/HS}} \in \mathbb{R}^{n \times n}, \qquad
(\tilde K_{\mathrm{IPR/HS}})_{ij} = \tilde K_{\mathrm{IPR/HS}}(h_i,h_j).
$$

The phase labels are assigned from the TFIM critical point $h_c=1$,

$$
y_i = \mathbf{1}[h_i \ge h_c], \qquad s_i = 2y_i - 1 \in \{-1,+1\},
$$

and the target label kernel is

$$
T_{ij}=s_i s_j.
$$

Both matrices are centered with the standard kernel-centering operation, implemented below with `sklearn.preprocessing.KernelCenterer`:

$$
\hat K_{\mathrm{IPR/HS}} = H\tilde K_{\mathrm{IPR/HS}}H, \qquad
\hat T = HTH, \qquad
H=I-\frac{1}{n}\mathbf{1}\mathbf{1}^{\top}.
$$

The plotted quantity is

$$
\mathrm{KTA}_{\mathrm{centered}}(\tilde K_{\mathrm{IPR/HS}},T)=
\frac{\langle \hat K_{\mathrm{IPR/HS}},\hat T\rangle_F}
{\|\hat K_{\mathrm{IPR/HS}}\|_F\,\|\hat T\|_F}.
$$

For binary labels, centering $T$ already accounts for label imbalance by subtracting the label mean before forming the centered target structure, so no additional class weighting is used here.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import KernelCenterer

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "scripts").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from data_config import available_thermal_betas
from plot_gram_matrices import compute_thermal_gram_matrix
from plot_ipr_curves import set_ipr_plot_style

L_values = [20, 50, 100]
kinds = ("ipr", "hs")
source = "processed"
hc = 1.0


In [2]:
def phase_labels_from_h(x, hc=1.0):
    return (np.asarray(x, dtype=float) >= float(hc)).astype(int)


def signed_phase_labels(y):
    return 2.0 * np.asarray(y, dtype=float) - 1.0


def center_matrix(A):
    A = np.asarray(A, dtype=float)
    return KernelCenterer().fit_transform(A)


def kernel_target_alignment(K, y):
    """Standard centered KTA using the ordinary Frobenius inner product."""
    K = np.asarray(K, dtype=float)
    y = np.asarray(y, dtype=int)
    if K.ndim != 2 or K.shape[0] != K.shape[1]:
        raise ValueError("KTA requires a square kernel matrix.")
    if K.shape[0] != y.size:
        raise ValueError("The number of labels must match the kernel matrix size.")
    if np.unique(y).size < 2:
        return np.nan

    signed = signed_phase_labels(y)
    target = np.outer(signed, signed)
    K_centered = center_matrix(K)
    target_centered = center_matrix(target)

    denom = np.linalg.norm(K_centered, ord="fro") * np.linalg.norm(target_centered, ord="fro")
    if denom == 0:
        return np.nan
    return float(np.sum(K_centered * target_centered) / denom)

In [ ]:
kta_rows = []
common_betas_by_L = {}

for L in L_values:
    available_beta_sets = [
        set(float(beta) for beta in available_thermal_betas(kind=kind, L=L, source=source))
        for kind in kinds
    ]
    betas = sorted(set.intersection(*available_beta_sets), reverse=True)
    if not betas:
        raise ValueError(f"No common beta values found for L={L} and kernels {kinds}.")
    common_betas_by_L[L] = betas

    for kind in kinds:
        for beta in betas:
            gram = compute_thermal_gram_matrix(
                kind=kind,
                L=L,
                beta=beta,
                source=source,
                normalize=True,
            )
            K = np.asarray(gram["K"], dtype=float)
            x = np.asarray(gram["x"], dtype=float)
            y = phase_labels_from_h(x, hc=hc)

            kta_rows.append(
                {
                    "L": int(L),
                    "kind": kind,
                    "beta": float(beta),
                    "standard_kta": kernel_target_alignment(K, y),
                    "n_phase_0": int(np.sum(y == 0)),
                    "n_phase_1": int(np.sum(y == 1)),
                }
            )

set_ipr_plot_style()
colors = {"ipr": "#1b9e77", "hs": "#d95f02"}
labels = {"ipr": "IPR", "hs": "HS"}
markers = {20: "o", 50: "s", 100: "^"}
linestyles = {"ipr": "-", "hs": "-"}
save_pdf = True
pdf_path = PROJECT_ROOT / "Paper Analysis" / "TFIM Thermal" / "Paper plots Thermal" / "thermal_centered_kta.pdf"

fig, ax = plt.subplots(figsize=(7.2, 4.8))

for kind in kinds:
    for L in L_values:
        rows = [
            row
            for row in kta_rows
            if row["L"] == int(L) and row["kind"] == kind
        ]
        beta_values = [row["beta"] for row in rows]
        values = [row["standard_kta"] for row in rows]
        ax.plot(
            beta_values,
            values,
            marker=markers.get(int(L), "o"),
            linestyle=linestyles.get(kind, "-"),
            color=colors.get(kind),
            label=f"{labels.get(kind, kind.upper())}, L={L}",
        )

all_betas = [beta for betas in common_betas_by_L.values() for beta in betas]
ax.axhline(0.0, color="0.4", lw=0.8, ls=":")
ax.set_xlim(max(all_betas), min(all_betas))
ax.set_ylim(0.4, 1.02)
ax.set_xlabel(r"$\beta$")
ax.set_ylabel("Centered KTA")
ax.tick_params(direction="in", which="both", top=True, right=True)
ax.minorticks_on()
ax.grid(which="major", alpha=0.20, lw=0.7)
ax.grid(which="minor", alpha=0.07, lw=0.5)
handles, legend_labels = ax.get_legend_handles_labels()
ax.legend(
    handles,
    legend_labels,
    loc="lower left",
    frameon=False,
    fontsize=14,
    ncol=2,
)
fig.tight_layout()
if save_pdf:
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    print(f"Saved figure to {pdf_path}")
plt.show()

kta_rows